In [ ]:
# 1 u
import os
import torch
import torch.nn as nn
import xarray as xr
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm

# ======================
# 路径配置
# ======================
DATA_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc"
BEST_MODEL_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/MSLP/1 u/subseasonal_model_u_best.pth"
NORM_PARAM_FILE = '/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/MSLP/1 u/norm_params.npz'

OUTPUT_NC = "pred_subseasonal_model_u_20260831.nc"

# ======================
# 输入变量
# ======================
input_vars = [
    *[f"mslp_hist_{i}" for i in range(20)],
    *[f"z850_hist_{i}" for i in range(10)],
    *[f"z500_z850_hist_{i}" for i in range(10)],
    *[f"q700_hist_{i}" for i in range(10)],
    *[f"divergence900_hist_{i}" for i in range(10)],
    *[f"pv900_hist_{i}" for i in range(10)],
    "pred_msl_month", "pred_t2m_month", "elevation"
]

# 输出变量名（模型输出通道）
target_vars = [
    "mslp_mon", "mslp_wed", "mslp_fri", "mslp_sun",
    "mslp_tue_next", "mslp_thu_next", "mslp_sat_next"
]

# ======================
# 加载数据（只用输入变量）
# ======================
ds = xr.open_dataset(DATA_PATH)
for var in input_vars:
    if ds[var].isnull().any():
        print(f"[NaN Warning] {var} 中存在 NaN，将填充为 0")
        ds[var] = ds[var].fillna(0)

# ======================
# 加载归一化参数
# ======================
params = np.load(NORM_PARAM_FILE)
x_mean, x_std = params['x_mean'], params['x_std']
y_mean, y_std = params['y_mean'], params['y_std']

# ======================
# 数据准备
# ======================
def prepare_input(ds):
    x_data = ds[input_vars].to_array().transpose('time', 'variable', 'latitude', 'longitude').values
    x_data = (x_data - x_mean[None, :, None, None]) / x_std[None, :, None, None]
    return np.nan_to_num(x_data).astype(np.float32)

X = prepare_input(ds)

# ======================
# 模型结构
# ======================
def center_crop(tensor, target_size):
    _, _, h, w = tensor.size()
    th, tw = target_size
    x1 = (h - th) // 2
    y1 = (w - tw) // 2
    return tensor[:, :, x1:x1+th, y1:y1+tw]

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_c)
        if in_c != out_c:
            self.residual = nn.Conv2d(in_c, out_c, 1)
        else:
            self.residual = nn.Identity()
    def forward(self, x):
        res = self.residual(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += res
        return self.relu(out)

class UNet_Res(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.enc1 = ResidualBlock(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ResidualBlock(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ResidualBlock(128, 256)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(256, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = ResidualBlock(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = ResidualBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = ResidualBlock(128, 64)
        self.output_layer = nn.Conv2d(64, out_channels, kernel_size=1)
    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)
        e3 = self.enc3(p2)
        p3 = self.pool3(e3)
        b = self.bottleneck(p3)
        u3 = self.up3(b)
        if u3.size()[2:] != e3.size()[2:]:
            e3 = center_crop(e3, u3.size()[2:])
        d3 = self.dec3(torch.cat([u3, e3], dim=1))
        u2 = self.up2(d3)
        if u2.size()[2:] != e2.size()[2:]:
            e2 = center_crop(e2, u2.size()[2:])
        d2 = self.dec2(torch.cat([u2, e2], dim=1))
        u1 = self.up1(d2)
        if u1.size()[2:] != e1.size()[2:]:
            e1 = center_crop(e1, u1.size()[2:])
        d1 = self.dec1(torch.cat([u1, e1], dim=1))
        out = self.output_layer(d1)
        if out.shape[2] != 121 or out.shape[3] != 240:
            out = F.interpolate(out, size=(121,240), mode='bilinear', align_corners=False)
        return out

# ======================
# 加载模型
# ======================
device = torch.device("cpu")
model = UNet_Res(len(input_vars), len(target_vars)).to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.eval()

# ======================
# 推理 + 反归一化
# ======================
Y_pred = []
with torch.no_grad():
    for i in tqdm(range(X.shape[0]), desc="Predicting"):
        x = torch.from_numpy(X[i:i+1]).to(device)
        pred = model(x).cpu().numpy()
        pred = pred * y_std[None, :, None, None] + y_mean[None, :, None, None]
        Y_pred.append(pred[0])
Y_pred = np.stack(Y_pred, axis=0)

# ======================
# 保存 NetCDF 文件
# ======================
output_ds = xr.Dataset(
    {var: (("time", "latitude", "longitude"), Y_pred[:, i]) for i, var in enumerate(target_vars)},
    coords={
        "time": ds.time.values,
        "latitude": ds.latitude.values,
        "longitude": ds.longitude.values,
    }
)
output_ds.to_netcdf(OUTPUT_NC)
print(f"✅ Saved prediction to {OUTPUT_NC}")


In [ ]:
# 2 SA
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only

import xarray as xr
import numpy as np
import torch
import torch.nn as nn

# ===========================
# 路径配置
# ===========================
IN_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc"
OUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_sa_20260831.nc"
CKPT_PATH = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/MSLP/2 SA/subseasonal_model_mslp_SpatialAttentionResNet9992-2.pth"

DEVICE = torch.device("cpu")

# ===========================
# 变量定义（完全不改）
# ===========================
input_vars = [
    *[f"mslp_hist_{i}" for i in range(20)],
    *[f"z850_hist_{i}" for i in range(10)],
    *[f"z500_z850_hist_{i}" for i in range(10)],
    *[f"q700_hist_{i}" for i in range(10)],
    *[f"divergence900_hist_{i}" for i in range(10)],
    *[f"pv900_hist_{i}" for i in range(10)],
    "pred_msl_month",
    "pred_t2m_month",
    "elevation"
]

target_vars = [
    "mslp_mon", "mslp_wed", "mslp_fri", "mslp_sun",
    "mslp_tue_next", "mslp_thu_next", "mslp_sat_next"
]

# ===========================
# 模型（与训练完全一致）
# ===========================
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        attn = self.sigmoid(self.conv(x_cat))
        return x * attn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.relu = nn.ReLU()
        self.attn = SpatialAttention()
        self.skip = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        identity = self.skip(x)
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out = self.attn(out)
        out += identity
        return self.relu(out)

class SpatialAttentionResNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64),
            ResidualBlock(64, 64),
            ResidualBlock(64, 32),
        )
        self.decoder = nn.Conv2d(32, out_channels, 1)

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = SpatialAttentionResNet(len(input_vars), len(target_vars)).to(DEVICE)
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()

# ===========================
# 读取数据
# ===========================
ds = xr.open_dataset(IN_FILE)

# NaN 处理
for v in input_vars:
    if v in ds:
        ds[v] = ds[v].fillna(0)

# ===========================
# 构造模型输入 (1, C, H, W)
# ===========================
X_list = []
for v in input_vars:
    da = ds[v]
    if "time" in da.dims:
        X_list.append(da.isel(time=0).values)
    else:
        X_list.append(da.values)

X = np.stack(X_list, axis=0)[None, ...]  # (1, C, lat, lon)

# ===========================
# 推理
# ===========================
with torch.no_grad():
    x = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    y_hat = model(x).cpu().numpy()[0]  # (lead, lat, lon)

# ===========================
# 保存 nc（每个 lead 单独变量）
# ===========================
data_vars = {name: (["time", "latitude", "longitude"], y_hat[i][None, ...])
             for i, name in enumerate(target_vars)}

ds_out = xr.Dataset(
    data_vars=data_vars,
    coords={
        "time": ds.time,
        "latitude": ds.latitude,
        "longitude": ds.longitude
    }
)

ds_out.to_netcdf(OUT_FILE)
print(f"✅ 单时刻推理完成，已保存：\n{OUT_FILE}")


In [ ]:
# 3 TC
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # CPU only

import xarray as xr
import numpy as np
import torch
import torch.nn as nn

# ===========================
# 配置路径
# ===========================
INPUT_FILE = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc'
MODEL_PATH = '/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/MSLP/3 TC/subseasonal_model_mslp_TransformerCNN-2.pth'
OUTPUT_FILE = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_tc_20260831.nc'

DEVICE = torch.device("cpu")

# ===========================
# 输入输出变量
# ===========================
input_vars = (
    [f'mslp_hist_{i}' for i in range(20)] +
    [f'z850_hist_{i}' for i in range(10)] +
    [f'z500_z850_hist_{i}' for i in range(10)] +
    [f'q700_hist_{i}' for i in range(10)] +
    [f'divergence900_hist_{i}' for i in range(10)] +
    [f'pv900_hist_{i}' for i in range(10)] +
    ['pred_msl_month', 'pred_t2m_month', 'elevation']
)

target_vars = [
    'mslp_mon', 'mslp_wed', 'mslp_fri', 'mslp_sun',
    'mslp_tue_next', 'mslp_thu_next', 'mslp_sat_next'
]

# ===========================
# 数据集
# ===========================
class SingleTimeDataset(torch.utils.data.Dataset):
    def __init__(self, ds, input_vars):
        arr_list = []
        for v in input_vars:
            da = ds[v]
            if 'time' in da.dims:
                arr_list.append(da.isel(time=0).values)
            else:
                arr_list.append(da.values)
        self.x = np.stack(arr_list, axis=0)[None, ...]  # (1, C, lat, lon)

    def __len__(self):
        return 1

    def __getitem__(self, idx):
        return torch.tensor(self.x[idx], dtype=torch.float32)

# ===========================
# 模型
# ===========================
class TransformerCNN(nn.Module):
    def __init__(self, in_channels, out_channels, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.patch = nn.Conv2d(in_channels, d_model, 3, padding=1)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=128)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.decoder = nn.Sequential(
            nn.Conv2d(d_model, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, out_channels, 1)
        )

    def forward(self, x):
        x = self.patch(x)
        B, C, H, W = x.shape
        x = x.flatten(2).permute(0, 2, 1)  # (B, H*W, C)
        x = self.transformer(x)
        x = x.permute(0, 2, 1).reshape(B, C, H, W)
        return self.decoder(x)

# ===========================
# 加载模型
# ===========================
model = TransformerCNN(len(input_vars), len(target_vars)).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# ===========================
# 读取数据
# ===========================
ds = xr.open_dataset(INPUT_FILE)

# 填充 NaN
for v in input_vars:
    if v in ds:
        ds[v] = ds[v].fillna(0)

dataset = SingleTimeDataset(ds, input_vars)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

# ===========================
# 推理
# ===========================
with torch.no_grad():
    for x in dataloader:
        x = x.to(DEVICE)
        pred = model(x).cpu().numpy()  # (1, lead, lat, lon)

pred = pred[0]  # 去掉 batch 维度

# ===========================
# 保存结果为单变量 nc
# ===========================
pred_ds = xr.Dataset(
    {var: (('time', 'latitude', 'longitude'), pred[i][None, ...]) for i, var in enumerate(target_vars)},
    coords={
        'time': ds['time'],
        'latitude': ds['latitude'],
        'longitude': ds['longitude']
    }
)

pred_ds.to_netcdf(OUTPUT_FILE)
print(f"✅ 推理完成，保存至 {OUTPUT_FILE}")


In [ ]:
# 4 upCG3
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""  # 禁用 GPU

import xarray as xr
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import numpy as np

# ===========================
# 配置
# ===========================
INPUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp-1/merged_with_pred/mslp_weekly_with_monthly_pred_with_elev.nc"
OUTPUT_FILE = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_upCG3_20260831.nc"
DEVICE = torch.device("cpu")

# 输入/输出变量
input_vars = (
    [f"mslp_hist_{i}" for i in range(20)] +
    [f"z850_hist_{i}" for i in range(10)] +
    [f"z500_z850_hist_{i}" for i in range(10)] +
    [f"q700_hist_{i}" for i in range(10)] +
    [f"divergence900_hist_{i}" for i in range(10)] +
    [f"pv900_hist_{i}" for i in range(10)] +
    ["pred_msl_month", "pred_t2m_month", "elevation"]
)
target_vars = [
    "mslp_mon", "mslp_wed", "mslp_fri", "mslp_sun",
    "mslp_tue_next", "mslp_thu_next", "mslp_sat_next"
]

# ===========================
# Dataset
# ===========================
class SubseasonalDataset(Dataset):
    def __init__(self, ds):
        # 对输入变量转成 array: (time, variable, lat, lon)
        self.x = ds[input_vars].to_array().transpose("time", "variable", "latitude", "longitude")
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, idx):
        return torch.tensor(self.x[idx].values, dtype=torch.float32)

# ===========================
# ConvGRU 模型
# ===========================
DROPOUT_RATE = 0.1
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 3 * hidden_dim, kernel_size, padding=padding)
        self.norm = nn.GroupNorm(3, 3 * hidden_dim)
    def forward(self, x, h_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates = self.norm(self.conv(combined))
        z, r, h_hat = torch.chunk(gates, 3, dim=1)
        z = torch.sigmoid(z)
        r = torch.sigmoid(r)
        h_hat = torch.tanh(r * h_hat)
        return (1 - z) * h_prev + z * h_hat

class ConvGRUNet(nn.Module):
    def __init__(self, in_ch, hid_ch, out_ch, gru_steps=3):
        super().__init__()
        self.gru_steps = gru_steps
        self.encoder = nn.Sequential(
            nn.Conv2d(in_ch, hid_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hid_ch, hid_ch, 3, padding=1)
        )
        self.encoder_norm = nn.GroupNorm(8, hid_ch)
        self.rnn = ConvGRUCell(hid_ch, hid_ch, 3)
        self.dropout = nn.Dropout(DROPOUT_RATE)
        self.decoder = nn.Conv2d(hid_ch, out_ch, 1)
    def forward(self, x):
        feat = self.encoder_norm(self.encoder(x))
        h = torch.zeros_like(feat)
        for _ in range(self.gru_steps):
            h = self.rnn(feat, h)
            h = h + feat
            h = self.dropout(h)
        return self.decoder(h)

# ===========================
# 模型加载
# ===========================
model_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/推理用/AZN/MSLP/4 upCG3/uplast_checkpoint_batch-2.pth"
model = ConvGRUNet(len(input_vars), 64, len(target_vars), gru_steps=3).to(DEVICE)
ckpt = torch.load(model_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

# ===========================
# 推理单时刻
# ===========================
ds = xr.open_dataset(INPUT_FILE)

# 填充缺失值
for v in input_vars:
    if v in ds.variables:
        ds[v] = ds[v].fillna(0)

# 标准化输入
for v in input_vars:
    if v in ds.variables:
        mean = ds[v].mean()
        std = ds[v].std()
        ds[v] = (ds[v] - mean) / (std if std != 0 else 1)

dataset = SubseasonalDataset(ds)
dataloader = DataLoader(dataset, batch_size=1)

with torch.no_grad():
    for x in dataloader:
        x = x.to(DEVICE)
        pred = model(x)  # [time, variable, lat, lon]

# ===========================
# 保存为指定格式
# ===========================
pred = pred.cpu().numpy()[0]  # 变量维度
data_vars = {v: (("time", "latitude", "longitude"), pred[i:i+1]) for i, v in enumerate(target_vars)}

ds_out = xr.Dataset(
    data_vars=data_vars,
    coords={
        "number": xr.DataArray([ds.number.values], dims=("time",)),
        "valid_time": xr.DataArray([ds.valid_time.values], dims=("time",)),
        "time": ds.time,
        "latitude": ds.latitude,
        "longitude": ds.longitude
    }
)

ds_out.to_netcdf(OUTPUT_FILE)
print(f"🎉 推理完成，保存到 {OUTPUT_FILE}")


In [ ]:
# 读取nc文件信息
import xarray as xr
# 定义文件路径
file_path = '/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_upCG3_20260831.nc'
# 打印原文件路径
print(f"文件路径: {file_path}")
# 打开NetCDF文件
ds = xr.open_dataset(file_path)
# 输出数据集的基本信息
print(ds)
# 如果需要查看数据集的变量列表，可以使用
#print(ds.variables)
# 如果需要查看数据集的维度，可以使用
#print(ds.dims)


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import math

# ===============================
# 文件路径
# ===============================
nc_file = "/Users/zhangnan/日常文件/ai/每周下载和预测/53 20260831/20260831-mslp/20260831-mslp 脚本/pred_subseasonal_model_upCG3_20260831.nc"

# ===============================
# 读取数据
# ===============================
ds = xr.open_dataset(nc_file)
variables = list(ds.data_vars)

# ===============================
# 拼图参数
# ===============================
n_vars = len(variables)
n_cols = 3  # 每行三个图
n_rows = math.ceil(n_vars / n_cols)
fig = plt.figure(figsize=(6*n_cols, 4*n_rows))

# ===============================
# 遍历变量绘制子图
# ===============================
for i, var_name in enumerate(variables):
    data = ds[var_name].squeeze('time')  # 去掉时间维度
    ax = plt.subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())
    
    # 绘制填色图
    im = data.plot.pcolormesh(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cmap='coolwarm',
        add_colorbar=True,
        cbar_kwargs={'label': var_name}
    )
    
    # 添加海岸线和边界
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    
    # 标题
    ax.set_title(f"{var_name} on {str(ds['time'].values[0])}")

plt.tight_layout()
plt.show()
